# How to Compute the Conjunctive Normal Form

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the *parser* that is found in the notebook `Propositional-Logic-Parser.ipynb`.

In [ ]:
import { LogicParser, Formula, Variable } from './PropositionalLogicParser';

Furthermore, we use the library `recursive-set`.  This library implements sets in a way such that the elements of sets are compared structurally and not by reference.

In [ ]:
import { RecursiveSet, Tuple, Value } from 'recursive-set';
type RS<T extends Value> = RecursiveSet<T>;

In [ ]:
function set<T extends Value>(...elements: T[]): RS<T> {
    return new RecursiveSet(...elements);
}

In [ ]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

In [ ]:
type Literal  = Variable 
              | Tuple<['¬', Variable]>;
type Clause   = RecursiveSet<Literal>;
type CNF      = RecursiveSet<Clause>;

In [ ]:
function parse(s: string): Formula {
    const parser = new LogicParser(s);
    return parser.parse();
}

The function `eliminateBiconditional(f)` takes a formula `f` from propositional logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$ (g \leftrightarrow h) \;\Leftrightarrow\; (g \rightarrow h) \wedge (h \rightarrow g) $$

In [ ]:
type Formula1 = Variable 
              | ['⊤' | '⊥']
              | ['¬', Formula1]
              | ['→' | '∧' | '∨', Formula1, Formula1];

In [ ]:
function eliminateBiconditional(f: Formula): Formula1 {
    'Eliminate the logical operator "↔" from f.'
    if (typeof f == 'string') {
        return f;
    }
    switch (f[0]) {
        case '↔': {
            const [, g, h] = f;
            return eliminateBiconditional(['∧', ['→', g, h], ['→', h, g]]);
        }
        case '⊤':
        case '⊥':
            return f;        
        case '¬': 
            return ['¬', eliminateBiconditional(f[1])];
        case '→':
        case '∧':
        case '∨': {
            const op = f[0];
            const g  = eliminateBiconditional(f[1]);
            const h  = eliminateBiconditional(f[2]);
            return [op, g, h];
        }
    }
}

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from propositional logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$ (g \rightarrow h) \;\Leftrightarrow\; (\neg g \vee h) $$

In [ ]:
type Formula2 = Variable 
              | ['⊤' | '⊥'] 
              | ['¬', Formula2]
              | ['∧' | '∨', Formula2, Formula2];

In [ ]:
function eliminateConditional(f: Formula1): Formula2 {
    if (typeof f == 'string') { // f is a variable
        return f; 
    }
    switch (f[0]) {
        case '⊤':
        case '⊥':
            return f;
        case '→': {
            const [, g, h] = f;
            return eliminateConditional(['∨', ['¬', g], h]);
        }
        case '¬': 
            return ['¬', eliminateConditional(f[1])];
        case '∧':
        case '∨': {
            const op = f[0];
            const g  = eliminateConditional(f[1]);
            const h  = eliminateConditional(f[2]);
            return [op, g, h];
        }
    }
}

In [ ]:
type NNF = Variable 
         | ['⊤' | '⊥'] 
         | ['¬', Variable] 
         | ['∧' | '∨', NNF, NNF];

The function $\texttt{nnf}(f)$ computes the *negation normal form* of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
</ol>
The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
     <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
</ol>

As the functions `nnf` and `neg` are mutually recursive, we need to define together them in a single cell.

In [ ]:
function nnf(f: Formula2): NNF {
    if (typeof f == 'string') {
        return f;
    }
    switch (f[0]) {
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            return neg(f[1]);
        }
        case '∧':
        case '∨': {
            const [op, g, h] = f;
            return [op, nnf(g), nnf(h)];
        }
    }
}

function neg(f: Formula2): NNF {
    if (typeof f == 'string') {
        return ['¬', f];
    }
    switch (f[0]) {
        case '⊤':
            return ['⊥'];
        case '⊥':
            return ['⊤'];
        case '¬': {
            return nnf(f[1]);
        }
        case '∧': {
            const [, g, h] = f;
            return ['∨', neg(g), neg(h)];
        }
        case '∨': {
            const [, g, h] = f;
            return ['∧', neg(g), neg(h)];
        }
    } 
}

The function $\texttt{cnf}(f)$ takes a formula $f$ that is in *negation normal form*, i.e. the negation operator is only applied to propositional variables and returns the *conjunctive normal form* of $f$ in *set notation*.  In order to achieve
this it uses the distributive law
$$ (f \wedge g) \vee (h \wedge k) \Leftrightarrow (f \vee h) \wedge (f \vee k) \wedge (g \vee h) \wedge (g \vee k). $$

In [ ]:
function cnf(f: NNF): CNF {
    if (typeof f == 'string') { 
        return set(set(f));
    }
    switch (f[0]) {
        case '⊤':
            return set();     
        case '⊥':
            return set(set());
        case '¬': {
            return set(set(tpl('¬', f[1])));
        }            
        default: {
            const [g, h] = [cnf(f[1]), cnf(f[2])]
            switch (f[0]) {
                case '∧': return g.union(h);
                case '∨': {
                    return g.cartesianProduct(h)
                            .map(([c1, c2]) => c1.union(c2));
                }
            }
        }
    }
}

The function `getComplement` computes the complement $\overline{l}$ of a literal $l$. 
If $p$ is a propositional variable, the complement is defined as follows:
 * $\overline{p} = \neg p$,
 * $\overline{\neg p} = p$.  

In [ ]:
function getComplement(l: Literal): Literal {
    if (typeof l == 'string') {
        return tpl('¬', l);
    } else {
        return l.get(1);
    }
}

The function $\texttt{isTrivial}(C)$ checks whether the clause $C$ is *trivial*.

In [ ]:
function isTrivial(clause: Clause): boolean {
    return clause.some(l => clause.has(getComplement(l)));
}

The function $\texttt{simplify}(Cs)$ takes a set of clauses and removes all trivial clauses from $Cs$.

In [ ]:
function simplify(clauses: CNF): CNF {
    return clauses.filter(c => !isTrivial(c));
}

The function $\texttt{normalize}$ takes a propositional formula $f$ and transforms $f$ into *conjunctive normal form*.  
Furthermore, trivial clausues are removed.

In [ ]:
function normalize(f: Formula): CNF {
    const n1 = eliminateBiconditional(f);
    const n2 = eliminateConditional(n1);
    const n3 = nnf(n2);
    const n4 = cnf(n3);
    return simplify(n4);
}

In [ ]:
function prettify(M: CNF): string {
    const clauseStrings: string[] = [];
    for (const clause of M) {
        const literalStrings: string[] = [];
        for (const lit of clause) {
            if (typeof lit == 'string') {
                literalStrings.push(lit);
            } else {
                literalStrings.push(`${lit.get(0)}${lit.get(1)}`);
            }
        }
        clauseStrings.push(`{${literalStrings.join(', ')}}`);
    }
    return `{${clauseStrings.join(', ')}}`;
}

In [ ]:
function test(s: string): string {
    const f = parse(s);
    console.log(`The knf of ${s} is:`);
    return prettify(normalize(f));
}

In [ ]:
test('(¬p → q) → (p → q) → q');

In [ ]:
test('(a → b) ↔ (¬a ∧ ¬b)');

In [ ]:
test('(p ∧ q → r) ∨ ¬r → ¬p');

In [ ]:
test('⊤');

In [ ]:
test('⊥');

In [ ]:
test('(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p');

In [ ]:
test("p → q");

In [ ]:
test("(p ∧ q) → r");

In [ ]:
test("p ↔ q");

In [ ]:
test("(p → q) ∧ (q → r)");

In [ ]:
test("¬(p ∨ q)");

In [ ]:
test('p ∧ p');